<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/solutions/03_derivative_linear_approx_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M3 微分・一次近似(notebook)

本文:[`03-derivative-linear-approx.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/math/03-derivative-linear-approx.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

> **これは解答つきの版である。** まず演習の版(`math/03_derivative_linear_approx.ipynb`)を自分で解き、答え合わせに使うこと。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-6):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

## 1〜2. 微分と数値微分

$f(x) = 3x^2 - 4x + 1$、$f'(x) = 6x - 4$。

**実装**:中心差分 $\dfrac{f(x+h) - f(x-h)}{2h}$ で数値微分を計算する関数 `num_diff` を書く。

In [ ]:
def f(x):
    return 3 * x**2 - 4 * x + 1

def f_prime(x):
    return 6 * x - 4

def num_diff(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

check("x=1 での微分", num_diff(f, 1.0), f_prime(1.0))
check("x=-2 での微分", num_diff(f, -2.0), f_prime(-2.0))

**予想**:$h$ を $10^{-1}$ から $10^{-12}$ まで小さくしていくと、前進差分と中心差分の誤差はそれぞれどうなるか(どんどん小さくなる?)。

- 予想:(ここに書く)

次のセルで、$f(x) = \sin x$、$x = 1$ での誤差を両対数グラフに描く。

In [ ]:
hs = 10.0 ** -np.arange(1, 13)
x0 = 1.0
err_fwd = [abs((np.sin(x0 + h) - np.sin(x0)) / h - np.cos(x0)) for h in hs]
err_ctr = [abs(num_diff(np.sin, x0, h) - np.cos(x0)) for h in hs]
plt.figure(figsize=(6, 3.5))
plt.loglog(hs, err_fwd, "o-", label="forward difference")
plt.loglog(hs, err_ctr, "s-", label="central difference")
plt.xlabel("h"); plt.ylabel("error"); plt.legend(); plt.grid(True, which="both", alpha=0.3); plt.show()

右側(大きい $h$)では、$h$ を小さくすると誤差が減る(前進差分は傾き1、中心差分は傾き2)。左側(小さすぎる $h$)では、浮動小数点の桁落ちで誤差が**増える**。中心差分では $h = 10^{-5}$ 前後が最もよい。

## 3. 偏微分と勾配

$g(x, y) = x^2 y + 3y$。$\nabla g = (2xy,\; x^2 + 3)$。

**実装**:各成分を1つずつ $\pm h$ だけ動かして中心差分をとり、勾配を返す関数 `num_grad` を書く。

In [ ]:
def g(v):
    x, y = v
    return x**2 * y + 3 * y

def num_grad(f, v, h=1e-5):
    v = np.asarray(v, dtype=float)
    grad = np.zeros_like(v)
    for i in range(len(v)):
        e = np.zeros_like(v); e[i] = h
        grad[i] = (f(v + e) - f(v - e)) / (2 * h)
    return grad

v0 = np.array([1.5, -2.0])
check("g の勾配", num_grad(g, v0), [2 * 1.5 * -2.0, 1.5**2 + 3])

## 4. 一次近似とヤコビ行列:2リンクアーム

リンクの長さ $l_1 = 1.0$、$l_2 = 0.8$。順運動学 `fk(q)` は用意してある。

**手計算**:$q = (0, \pi/2)$ のとき、手先の位置と $J$ を求める。

- 手計算:(ここに書く)

**実装**:本文の式どおりにヤコビ行列を返す関数 `jac(q)` を書く。

In [ ]:
l1, l2 = 1.0, 0.8

def fk(q):
    q1, q2 = q
    return np.array([l1 * np.cos(q1) + l2 * np.cos(q1 + q2),
                     l1 * np.sin(q1) + l2 * np.sin(q1 + q2)])

def jac(q):
    q1, q2 = q
    s1, c1 = np.sin(q1), np.cos(q1)
    s12, c12 = np.sin(q1 + q2), np.cos(q1 + q2)
    return np.array([[-l1 * s1 - l2 * s12, -l2 * s12],
                     [ l1 * c1 + l2 * c12,  l2 * c12]])

check("手先の位置 q=(0, π/2)", fk([0.0, np.pi / 2]), [1.0, 0.8])
check("J(0, π/2)", jac([0.0, np.pi / 2]), [[-0.8, -0.8], [1.0, 0.0]])

**数値微分で確かめる**:ヤコビ行列の列 $j$ は、関節 $j$ だけを動かしたときの手先の変化の割合である。

**実装**:中心差分でヤコビ行列を求める関数 `num_jac(f, q)` を書き、`jac` と比べる(いろいろな $q$ で)。

In [ ]:
def num_jac(f, q, h=1e-5):
    q = np.asarray(q, dtype=float)
    m, n = len(f(q)), len(q)
    J = np.zeros((m, n))
    for j in range(n):
        e = np.zeros(n); e[j] = h
        J[:, j] = (f(q + e) - f(q - e)) / (2 * h)
    return J

rng = np.random.default_rng(0)
for _ in range(3):
    q = rng.uniform(-np.pi, np.pi, size=2)
    check(f"jac と num_jac が一致 q={np.round(q, 2)}", jac(q), num_jac(fk, q))

**一次近似の誤差**:$\Delta q$ を小さくしていくと、$\lVert f(q + \Delta q) - f(q) - J\Delta q \rVert$ はどう小さくなるか。

**予想**:$\Delta q$ を 1/10 にすると、誤差は何分の1になるか。

- 予想:(ここに書く)

In [ ]:
q = np.array([0.4, 1.1])
d = np.array([1.0, -0.5]) / np.sqrt(1.25)        # 動かす方向(長さ 1)
scales = 10.0 ** -np.arange(0, 6)
errs = [np.linalg.norm(fk(q + s * d) - fk(q) - jac(q) @ (s * d)) for s in scales]
for s, e in zip(scales, errs):
    print(f"|Δq| = {s:.0e}:  一次近似の誤差 {e:.2e}")
plt.figure(figsize=(5, 3.5))
plt.loglog(scales, errs, "o-", label="error of linear approx.")
plt.loglog(scales, scales**2, "k--", label="|dq|^2 (slope 2)")
plt.xlabel("|dq|"); plt.ylabel("error"); plt.legend(); plt.grid(True, which="both", alpha=0.3); plt.show()

## 5. 勾配法

$x \leftarrow x - \eta f'(x)$ をくり返して、$f(x) = 3x^2 - 4x + 1$ の最小値($x = 2/3$)を探す。

**予想**:$\eta = 0.05, 0.3, 0.34$ のそれぞれで、100回くり返したあとの $x$ はどうなるか。

- 予想:(ここに書く)

**実装**:勾配法の関数 `gradient_descent` を書く。

In [ ]:
def gradient_descent(grad, x0, eta, steps):
    xs = [float(x0)]
    for _ in range(steps):
        xs.append(xs[-1] - eta * grad(xs[-1]))
    return np.array(xs)

check("eta=0.05 で 2/3 に収束", gradient_descent(f_prime, 3.0, 0.05, 200)[-1], 2 / 3)

plt.figure(figsize=(6, 3.5))
for eta in [0.05, 0.3, 0.34]:
    xs = gradient_descent(f_prime, 3.0, eta, 30)
    plt.plot(xs, marker=".", label=f"eta={eta}")
plt.axhline(2 / 3, color="k", ls="--"); plt.ylim(-6, 8)
plt.xlabel("step"); plt.ylabel("x"); plt.legend(); plt.show()

$\eta = 0.3$ では最小値の両側を行ったり来たりしながら近づき、$\eta = 0.34$ では発散する。この関数では $\eta < 1/3$ が収束の条件である(本文 §5)。

## 6. 発展:一次近似で手先を目標に近づける(数値逆運動学の入口)

目標 $x^\ast$ に対し、誤差 $e = x^\ast - f(q)$ を求め、$J(q)\,\Delta q = e$ を解いて $q \leftarrow q + \Delta q$ とする。これをくり返す(K4 で詳しく扱う)。

In [ ]:
target = np.array([0.6, 1.2])
q = np.array([0.3, 0.5])
for k in range(8):
    e = target - fk(q)
    print(f"step {k}: 手先 {np.round(fk(q), 4)}  誤差 {np.linalg.norm(e):.2e}")
    q = q + np.linalg.solve(jac(q), e)
check("順運動学に戻すと目標に一致", fk(q), target)

最初の数ステップでは、誤差がかえって増えることがある。一次近似は $q$ の近くでしか当たらないので、遠くの目標に一度に向かうと行き過ぎるからである。目標に近づくと、誤差は1ステップごとに桁違いに小さくなる。最後の `check` のように、**求めた関節角は順運動学に戻して確かめる**。

## 7. 条件を変える

1. 6 で、目標をアームが届かない点(例:`[2.5, 0.0]`。$l_1 + l_2 = 1.8$)にすると何が起きるか。「計算が収束しない」ことと「解が存在しない」ことは区別できるか
2. 6 で、初期値 `q` を変えると、別の関節角に収束することがある。手先は同じなのに関節角が違うのはなぜか(K2:逆運動学の複数の解)
3. 4 で、$q = (0.4, 0)$(腕をまっすぐ伸ばした姿勢)の $J$ の行列式 `np.linalg.det(jac(q))` を計算する。0 になるのはなぜか

## 8. 説明(自分の言葉で書く)

1. 数値微分の $h$ を小さくしすぎると誤差が増えるのはなぜか
   - (ここに書く)
2. ヤコビ行列の列 $j$ は何を表すか
   - (ここに書く)
3. 勾配法の学習率が大きすぎると何が起きるか
   - (ここに書く)